In [3]:
!pip install "numpy<2" scikit-surprise
import pandas as pd
import numpy as np
import subprocess
import io
import pickle
import time
import gc
from pathlib import Path
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split, GridSearchCV, cross_validate
from surprise import accuracy
import matplotlib.pyplot as plt
import seaborn as sns
print("hello world :)")

Defaulting to user installation because normal site-packages is not writeable


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# # 2. CHARGEMENT DES DONNÉES DEPUIS GOOGLE DRIVE

# %%
def load_ratings_from_gdrive(num_files=20):
    """
    Charge les fichiers ratings depuis Google Drive
    """
    print(f"📥 Chargement de {num_files} fichiers Parquet...")
    
    # Lister les fichiers
    result = subprocess.run(
        ["rclone", "lsf", "gdrive:movie_data_25m/parquet/"],
        capture_output=True, text=True
    )
    parquet_files = [f.strip() for f in result.stdout.split('\n') if f][:num_files]
    
    all_data = []
    total_rows = 0
    
    for i, filename in enumerate(parquet_files):
        print(f"   • {i+1}/{len(parquet_files)}: {filename}", end="")
        
        # Télécharger
        remote_path = f"gdrive:movie_data_25m/parquet/{filename}"
        result = subprocess.run(["rclone", "cat", remote_path], capture_output=True)
        
        # Lire
        df = pd.read_parquet(io.BytesIO(result.stdout))
        all_data.append(df)
        total_rows += len(df)

        print(f" → {len(df):,} lignes")
        
        if (i+1) % 5 == 0:
            gc.collect()
    
    # Combiner
    df_ratings = pd.concat(all_data, ignore_index=True)
    
    print(f"\n✅ Total chargé: {len(df_ratings):,} lignes")
    
    return df_ratings
   
        

In [3]:
# Charger les données
print("🎯 CHARGEMENT DES DONNÉES")
print("="*60)

df_ratings = load_ratings_from_gdrive(num_files=20)  # 20 fichiers = ~1M lignes



🎯 CHARGEMENT DES DONNÉES
📥 Chargement de 20 fichiers Parquet...
   • 1/20: chunk_00000.parquet → 60,000 lignes
   • 2/20: chunk_00001.parquet → 60,000 lignes
   • 3/20: chunk_00002.parquet → 60,000 lignes
   • 4/20: chunk_00003.parquet → 60,000 lignes
   • 5/20: chunk_00004.parquet → 60,000 lignes
   • 6/20: chunk_00005.parquet → 60,000 lignes
   • 7/20: chunk_00006.parquet → 60,000 lignes
   • 8/20: chunk_00007.parquet → 60,000 lignes
   • 9/20: chunk_00008.parquet → 60,000 lignes
   • 10/20: chunk_00009.parquet → 60,000 lignes
   • 11/20: chunk_00010.parquet → 60,000 lignes
   • 12/20: chunk_00011.parquet → 60,000 lignes
   • 13/20: chunk_00012.parquet → 60,000 lignes
   • 14/20: chunk_00013.parquet → 60,000 lignes
   • 15/20: chunk_00014.parquet → 60,000 lignes
   • 16/20: chunk_00015.parquet → 60,000 lignes
   • 17/20: chunk_00016.parquet → 60,000 lignes
   • 18/20: chunk_00017.parquet → 60,000 lignes
   • 19/20: chunk_00018.parquet → 60,000 lignes
   • 20/20: chunk_00019.parquet →

In [ ]:
# Charger movies (local)
df_movies = pd.read_parquet("/mnt/c/Users/SOUMI/movie-recommendation-system/data/processed/parquet_export/movies.parquet")
df_users = pd.read_parquet("/mnt/c/Users/SOUMI/movie-recommendation-system/data/processed/parquet_export/users.parquet")

print(f"\n📊 Statistiques:")
print(f"   • Ratings: {len(df_ratings):,}")
print(f"   • Films: {len(df_movies):,}")
print(f"   • Utilisateurs: {len(df_users):,}")

In [ ]:
# ==========================================
# NETTOYAGE COMPLET DES DONNÉES POUR SVD
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

def prepare_svd_data_complete(df, 
                               min_user_ratings=5, 
                               min_item_ratings=5,
                               rating_scale=(0.5, 5.0),
                               save_mappings=True,
                               output_dir='models'):
    """
    Nettoyage complet et préparation des données pour SVD
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame avec colonnes user_key, movie_key, rating
    min_user_ratings : int
        Nombre minimum de ratings par utilisateur
    min_item_ratings : int
        Nombre minimum de ratings par film
    rating_scale : tuple
        (min, max) des notes valides
    save_mappings : bool
        Sauvegarder les mappings d'IDs
    output_dir : str
        Dossier pour sauvegarder les mappings
        
    Returns:
    --------
    df_clean : DataFrame
        Données nettoyées prêtes pour SVD
    stats : dict
        Statistiques du nettoyage
    """
    
    print("\n" + "="*70)
    print("🧹 NETTOYAGE COMPLET DES DONNÉES POUR SVD")
    print("="*70)
    
    # Créer dossier de sortie
    Path(output_dir).mkdir(exist_ok=True)
    
    # Statistiques de nettoyage
    stats = {
        'initial': len(df),
        'steps': []
    }
    
    # ==========================================
    # ÉTAPE 1 : SÉLECTION DES COLONNES
    # ==========================================
    print("\n📋 ÉTAPE 1 : Sélection des colonnes")
    
    required_cols = ['user_key', 'movie_key', 'rating']
    
    # Vérifier que les colonnes existent
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Colonnes manquantes: {missing_cols}")
        print(f"   Colonnes disponibles: {df.columns.tolist()}")
        return None, None
    
    df = df[required_cols].copy()
    df.columns = ['user', 'item', 'rating']
    
    print(f"✅ Colonnes sélectionnées: user, item, rating")
    print(f"   Lignes initiales: {len(df):,}")
    
    # ==========================================
    # ÉTAPE 2 : VALEURS MANQUANTES
    # ==========================================
    print("\n🔍 ÉTAPE 2 : Gestion des valeurs manquantes")
    
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(f"⚠️  Valeurs manquantes détectées:")
        for col, count in null_counts.items():
            if count > 0:
                print(f"   • {col}: {count:,} ({count/len(df)*100:.2f}%)")
        
        before = len(df)
        df = df.dropna()
        removed = before - len(df)
        stats['steps'].append(('null_values', removed))
        print(f"   ✅ {removed:,} lignes supprimées")
    else:
        print("✅ Aucune valeur manquante")
        stats['steps'].append(('null_values', 0))
    
    # ==========================================
    # ÉTAPE 3 : NOTES INVALIDES
    # ==========================================
    print(f"\n⭐ ÉTAPE 3 : Validation des notes ({rating_scale[0]} à {rating_scale[1]})")
    
    before = len(df)
    df = df[(df['rating'] >= rating_scale[0]) & (df['rating'] <= rating_scale[1])]
    removed = before - len(df)
    stats['steps'].append(('invalid_ratings', removed))
    
    if removed > 0:
        print(f"⚠️  Notes invalides: {removed:,} supprimées ({removed/before*100:.2f}%)")
    else:
        print("✅ Toutes les notes sont valides")
    
    # ==========================================
    # ÉTAPE 4 : DOUBLONS
    # ==========================================
    print("\n🔄 ÉTAPE 4 : Suppression des doublons")
    
    before = len(df)
    df = df.drop_duplicates(subset=['user', 'item'], keep='last')
    removed = before - len(df)
    stats['steps'].append(('duplicates', removed))
    
    if removed > 0:
        print(f"⚠️  Doublons: {removed:,} supprimés ({removed/before*100:.2f}%)")
    else:
        print("✅ Aucun doublon détecté")
    
    # ==========================================
    # ÉTAPE 5 : FILTRAGE COLD START ITÉRATIF
    # ==========================================
    print(f"\n❄️  ÉTAPE 5 : Filtrage cold start (≥{min_user_ratings} ratings)")
    print("   Mode: Itératif (users et items)")
    
    initial_cold_start = len(df)
    prev_len = 0
    iteration = 0
    max_iterations = 10
    
    while len(df) != prev_len and iteration < max_iterations:
        prev_len = len(df)
        iteration += 1
        
        # Filtrer utilisateurs
        user_counts = df['user'].value_counts()
        active_users = user_counts[user_counts >= min_user_ratings].index
        df = df[df['user'].isin(active_users)]
        
        # Filtrer items
        item_counts = df['item'].value_counts()
        active_items = item_counts[item_counts >= min_item_ratings].index
        df = df[df['item'].isin(active_items)]
        
        users_remaining = df['user'].nunique()
        items_remaining = df['item'].nunique()
        
        print(f"   Iteration {iteration}: {len(df):,} lignes | "
              f"{users_remaining:,} users | {items_remaining:,} items")
    
    removed = initial_cold_start - len(df)
    stats['steps'].append(('cold_start', removed))
    
    if iteration >= max_iterations:
        print(f"⚠️  Arrêt après {max_iterations} itérations (limite atteinte)")
    else:
        print(f"✅ Convergence après {iteration} itérations")
    
    print(f"   Lignes supprimées: {removed:,} ({removed/initial_cold_start*100:.2f}%)")
    
    # ==========================================
    # ÉTAPE 6 : STATISTIQUES AVANT FACTORIZATION
    # ==========================================
    print("\n📊 ÉTAPE 6 : Statistiques avant normalisation")
    
    n_users = df['user'].nunique()
    n_items = df['item'].nunique()
    n_ratings = len(df)
    sparsity = 1 - (n_ratings / (n_users * n_items))
    
    print(f"   • Utilisateurs uniques: {n_users:,}")
    print(f"   • Films uniques: {n_items:,}")
    print(f"   • Ratings totaux: {n_ratings:,}")
    print(f"   • Sparsité: {sparsity*100:.2f}%")
    print(f"   • Ratings/user (moy): {n_ratings/n_users:.1f}")
    print(f"   • Ratings/item (moy): {n_ratings/n_items:.1f}")
    
    # ==========================================
    # ÉTAPE 7 : NORMALISATION DES IDs
    # ==========================================
    print("\n🔢 ÉTAPE 7 : Normalisation des IDs (factorization)")
    
    # Sauvegarder les IDs originaux
    user_original = df['user'].values.copy()
    item_original = df['item'].values.copy()
    
    # Factorize
    df['user'], user_inverse = pd.factorize(df['user'])
    df['item'], item_inverse = pd.factorize(df['item'])
    
    print(f"✅ IDs normalisés:")
    print(f"   • Users: {df['user'].min()} → {df['user'].max()}")
    print(f"   • Items: {df['item'].min()} → {df['item'].max()}")
    
    # ==========================================
    # ÉTAPE 8 : SAUVEGARDE DES MAPPINGS
    # ==========================================
    if save_mappings:
        print("\n💾 ÉTAPE 8 : Sauvegarde des mappings")
        
        mappings = {
            # Original → Factorized
            'user_to_idx': dict(zip(user_inverse, range(len(user_inverse)))),
            'item_to_idx': dict(zip(item_inverse, range(len(item_inverse)))),
            
            # Factorized → Original
            'idx_to_user': dict(enumerate(user_inverse)),
            'idx_to_item': dict(enumerate(item_inverse)),
            
            # Métadonnées
            'n_users': len(user_inverse),
            'n_items': len(item_inverse),
            'rating_scale': rating_scale
        }
        
        mapping_path = Path(output_dir) / 'id_mappings.pkl'
        with open(mapping_path, 'wb') as f:
            pickle.dump(mappings, f)
        
        print(f"✅ Mappings sauvegardés: {mapping_path}")
        print(f"   • {len(user_inverse):,} users")
        print(f"   • {len(item_inverse):,} items")
        
        # Exemple de mapping
        example_user_orig = user_inverse[0]
        example_user_new = mappings['user_to_idx'][example_user_orig]
        print(f"\n   Exemple: User {example_user_orig} → {example_user_new}")
    
    # ==========================================
    # ÉTAPE 9 : STATISTIQUES FINALES
    # ==========================================
    print("\n" + "="*70)
    print("📈 RÉSUMÉ DU NETTOYAGE")
    print("="*70)
    
    stats['final'] = len(df)
    stats['users_final'] = df['user'].nunique()
    stats['items_final'] = df['item'].nunique()
    stats['sparsity'] = sparsity
    
    print(f"\n🔢 STATISTIQUES:")
    print(f"   • Lignes initiales:  {stats['initial']:,}")
    print(f"   • Lignes finales:    {stats['final']:,}")
    print(f"   • Conservation:      {stats['final']/stats['initial']*100:.1f}%")
    print(f"   • Users finaux:      {stats['users_final']:,}")
    print(f"   • Items finaux:      {stats['items_final']:,}")
    print(f"   • Sparsité:          {stats['sparsity']*100:.2f}%")
    
    print(f"\n🗑️  SUPPRESSIONS PAR ÉTAPE:")
    total_removed = 0
    for step_name, removed in stats['steps']:
        if removed > 0:
            print(f"   • {step_name:20s}: {removed:,} lignes")
            total_removed += removed
    print(f"   {'TOTAL':20s}: {total_removed:,} lignes")
    
    print(f"\n📊 DISTRIBUTION DES NOTES:")
    rating_dist = df['rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        print(f"   • {rating:.1f} ⭐: {count:,} ({count/len(df)*100:.1f}%)")
    
    print(f"\n✅ Note moyenne: {df['rating'].mean():.3f}")
    print(f"✅ Note médiane: {df['rating'].median():.1f}")
    
    print("\n" + "="*70)
    print("✅ NETTOYAGE TERMINÉ")
    print("="*70)
    
    return df, stats


# ==========================================
# FONCTION DE VISUALISATION
# ==========================================

def visualize_cleaned_data(df, stats=None):
    """
    Visualisations des données nettoyées
    """
    print("\n📊 Génération des visualisations...")
    
    fig = plt.figure(figsize=(16, 10))
    
    # Layout: 2 rows, 3 columns
    
    # ==========================================
    # ROW 1: Distributions
    # ==========================================
    
    # 1. Distribution des notes
    ax1 = plt.subplot(2, 3, 1)
    df['rating'].hist(bins=20, edgecolor='black', alpha=0.7)
    ax1.axvline(df['rating'].mean(), color='red', linestyle='--', 
                label=f'Moyenne: {df["rating"].mean():.2f}')
    ax1.set_title('Distribution des notes', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Note')
    ax1.set_ylabel('Fréquence')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # 2. Ratings par utilisateur
    ax2 = plt.subplot(2, 3, 2)
    user_ratings = df.groupby('user').size()
    ax2.hist(user_ratings, bins=50, edgecolor='black', alpha=0.7, color='orange')
    ax2.axvline(user_ratings.mean(), color='red', linestyle='--',
                label=f'Moyenne: {user_ratings.mean():.1f}')
    ax2.set_title('Ratings par utilisateur', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Nombre de ratings')
    ax2.set_ylabel('Nombre d\'utilisateurs')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # 3. Ratings par film
    ax3 = plt.subplot(2, 3, 3)
    item_ratings = df.groupby('item').size()
    ax3.hist(item_ratings, bins=50, edgecolor='black', alpha=0.7, color='green')
    ax3.axvline(item_ratings.mean(), color='red', linestyle='--',
                label=f'Moyenne: {item_ratings.mean():.1f}')
    ax3.set_title('Ratings par film', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Nombre de ratings')
    ax3.set_ylabel('Nombre de films')
    ax3.legend()
    ax3.grid(alpha=0.3)
    
    # ==========================================
    # ROW 2: Analyses avancées
    # ==========================================
    
    # 4. Top 10 notes les plus fréquentes
    ax4 = plt.subplot(2, 3, 4)
    rating_counts = df['rating'].value_counts().sort_index()
    ax4.bar(rating_counts.index, rating_counts.values, edgecolor='black', alpha=0.7)
    ax4.set_title('Fréquence des notes', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Note')
    ax4.set_ylabel('Nombre de ratings')
    ax4.grid(alpha=0.3, axis='y')
    
    # 5. Boxplot ratings par utilisateur
    ax5 = plt.subplot(2, 3, 5)
    sample_users = df['user'].unique()[:100]  # 100 premiers users
    sample_df = df[df['user'].isin(sample_users)]
    sample_df.boxplot(column='rating', by='user', ax=ax5, showfliers=False)
    ax5.set_title('Variation des notes (100 premiers users)', fontsize=12, fontweight='bold')
    ax5.set_xlabel('Utilisateur')
    ax5.set_ylabel('Note')
    plt.suptitle('')  # Supprimer titre auto
    
    # 6. Statistiques textuelles
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')
    
    stats_text = f"""
    📊 STATISTIQUES FINALES
    {'─'*35}
    
    📈 Données:
       • Ratings:        {len(df):,}
       • Utilisateurs:   {df['user'].nunique():,}
       • Films:          {df['item'].nunique():,}
    
    ⭐ Notes:
       • Moyenne:        {df['rating'].mean():.3f}
       • Médiane:        {df['rating'].median():.1f}
       • Std:            {df['rating'].std():.3f}
    
    👥 Utilisateurs:
       • Ratings/user:   {len(df)/df['user'].nunique():.1f}
       • Min ratings:    {user_ratings.min()}
       • Max ratings:    {user_ratings.max():,}
    
    🎬 Films:
       • Ratings/film:   {len(df)/df['item'].nunique():.1f}
       • Min ratings:    {item_ratings.min()}
       • Max ratings:    {item_ratings.max():,}
    
    📉 Sparsité:        {(1 - len(df)/(df['user'].nunique()*df['item'].nunique()))*100:.2f}%
    """
    
    ax6.text(0.1, 0.95, stats_text, fontsize=10, verticalalignment='top',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Visualisations générées")


# ==========================================
# UTILISATION
# ==========================================

# Appeler la fonction de nettoyage
df_clean, cleaning_stats = prepare_svd_data_complete(
    df_ratings,
    min_user_ratings=5,
    min_item_ratings=5,
    rating_scale=(0.5, 5.0),
    save_mappings=True,
    output_dir='models'
)

# Visualiser
if df_clean is not None:
    visualize_cleaned_data(df_clean, cleaning_stats)
    
    print("\n✅ Données prêtes pour l'entraînement SVD !")
    print(f"   DataFrame: df_clean ({len(df_clean):,} lignes)")
    print(f"   Mappings: models/id_mappings.pkl")


🧹 NETTOYAGE COMPLET DES DONNÉES POUR SVD

📋 ÉTAPE 1 : Sélection des colonnes
✅ Colonnes sélectionnées: user, item, rating
   Lignes initiales: 1,200,000

🔍 ÉTAPE 2 : Gestion des valeurs manquantes
✅ Aucune valeur manquante

⭐ ÉTAPE 3 : Validation des notes (0.5 à 5.0)
✅ Toutes les notes sont valides

🔄 ÉTAPE 4 : Suppression des doublons
✅ Aucun doublon détecté

❄️  ÉTAPE 5 : Filtrage cold start (≥5 ratings)
   Mode: Itératif (users et items)
   Iteration 1: 1,176,549 lignes | 7,900 users | 10,775 items
   Iteration 2: 1,176,549 lignes | 7,900 users | 10,775 items
✅ Convergence après 2 itérations
   Lignes supprimées: 23,451 (1.95%)

📊 ÉTAPE 6 : Statistiques avant normalisation
   • Utilisateurs uniques: 7,900
   • Films uniques: 10,775
   • Ratings totaux: 1,176,549
   • Sparsité: 98.62%
   • Ratings/user (moy): 148.9
   • Ratings/item (moy): 109.2

🔢 ÉTAPE 7 : Normalisation des IDs (factorization)
✅ IDs normalisés:
   • Users: 0 → 7899
   • Items: 0 → 10774

💾 ÉTAPE 8 : Sauvegarde de

In [8]:
# # 4. DIVISION TRAIN/TEST

# %%
# Préparer pour Surprise
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_clean[['user', 'item', 'rating']], reader)

# Division 80/20
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"📊 Division des données:")
print(f"   • nombre de linge dans data {len(df_clean)} .")
print(f"   • Train: {trainset.n_ratings:,} notes")
print(f"   • Test: {len(testset):,} notes")
print(f"   • Ratio: 80/20")

📊 Division des données:
nombre de linge dans data 1176549 .
   • Train: 941,239 notes
   • Test: 235,310 notes
   • Ratio: 80/20


In [ ]:
# # 5. GRID SEARCH POUR TROUVER LES MEILLEURS PARAMÈTRES
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV
# %%
print("\n🎯 GRID SEARCH")
print("="*60)

# Définir la grille de paramètres
param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [10, 20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.04]
}

print(f"📊 Grille de recherche:")
print(f"   • {len(param_grid['n_factors'])} × {len(param_grid['n_epochs'])} × {len(param_grid['lr_all'])} × {len(param_grid['reg_all'])} = {len(param_grid['n_factors']) * len(param_grid['n_epochs']) * len(param_grid['lr_all']) * len(param_grid['reg_all'])} combinaisons")

# Grid Search avec validation croisée
gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse', 'mae'],
    cv=3,  # 3-fold cross-validation
    n_jobs=-1,  # Utiliser tous les CPUs
    joblib_verbose=1
)

print("\n⏳ Recherche en cours (peut prendre 10-15 minutes)...")
start_time = time.time()

gs.fit(data)

grid_time = time.time() - start_time

# Résultats
print(f"\n✅ Grid Search terminé en {grid_time/60:.1f} minutes")
print(f"\n🏆 MEILLEURS PARAMÈTRES:")
print(f"   • RMSE: {gs.best_score['rmse']:.4f}")
print(f"   • MAE: {gs.best_score['mae']:.4f}")
print(f"   • n_factors: {gs.best_params['rmse']['n_factors']}")
print(f"   • n_epochs: {gs.best_params['rmse']['n_epochs']}")
print(f"   • lr_all: {gs.best_params['rmse']['lr_all']}")
print(f"   • reg_all: {gs.best_params['rmse']['reg_all']}")


🎯 GRID SEARCH
📊 Grille de recherche:
   • 3 × 3 × 2 × 2 = 36 combinaisons


NameError: name 'GridSearchCV' is not defined

In [ ]:
# # 6. ANALYSE DES RÉSULTATS GRID SEARCH

# %%
# Convertir les résultats en DataFrame
results_df = pd.DataFrame(gs.cv_results)

# Trier par RMSE
results_df = results_df.sort_values('mean_test_rmse')

print("\n📊 TOP 10 MEILLEURES COMBINAISONS:")
print(results_df[['params', 'mean_test_rmse', 'std_test_rmse']].head(10).to_string())

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Effet de n_factors
for n_factors in param_grid['n_factors']:
    subset = results_df[results_df['param_n_factors'] == n_factors]
    axes[0,0].plot(subset['param_n_epochs'], subset['mean_test_rmse'], 
                   marker='o', label=f'factors={n_factors}')
axes[0,0].set_xlabel('n_epochs')
axes[0,0].set_ylabel('RMSE')
axes[0,0].set_title('Impact de n_factors et n_epochs')
axes[0,0].legend()
axes[0,0].grid(True)

# Effet de lr_all
for lr in param_grid['lr_all']:
    subset = results_df[results_df['param_lr_all'] == lr]
    axes[0,1].plot(subset['param_n_factors'], subset['mean_test_rmse'], 
                   marker='o', label=f'lr={lr}')
axes[0,1].set_xlabel('n_factors')
axes[0,1].set_ylabel('RMSE')
axes[0,1].set_title('Impact de lr_all')
axes[0,1].legend()
axes[0,1].grid(True)

# Effet de reg_all
for reg in param_grid['reg_all']:
    subset = results_df[results_df['param_reg_all'] == reg]
    axes[1,0].plot(subset['param_n_factors'], subset['mean_test_rmse'], 
                   marker='o', label=f'reg={reg}')
axes[1,0].set_xlabel('n_factors')
axes[1,0].set_ylabel('RMSE')
axes[1,0].set_title('Impact de reg_all')
axes[1,0].legend()
axes[1,0].grid(True)

# Distribution des RMSE
axes[1,1].hist(results_df['mean_test_rmse'], bins=20, edgecolor='black')
axes[1,1].axvline(gs.best_score['rmse'], color='r', linestyle='--', 
                  label=f'Best: {gs.best_score["rmse"]:.4f}')
axes[1,1].set_xlabel('RMSE')
axes[1,1].set_ylabel('Nombre de combinaisons')
axes[1,1].set_title('Distribution des RMSE')
axes[1,1].legend()
axes[1,1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# # 7. ENTRAÎNEMENT DU MODÈLE FINAL

# %%
print("\n🚀 ENTRAÎNEMENT DU MODÈLE FINAL")
print("="*60)

# Utiliser les meilleurs paramètres
best_params = gs.best_params['rmse']

print(f"📊 Paramètres optimaux:")
print(f"   • n_factors: {best_params['n_factors']}")
print(f"   • n_epochs: {best_params['n_epochs']}")
print(f"   • lr_all: {best_params['lr_all']}")
print(f"   • reg_all: {best_params['reg_all']}")

# Créer et entraîner le modèle final
algo = SVD(**best_params, random_state=42)

# Entraîner sur toutes les données
trainset_full = data.build_full_trainset()

start_time = time.time()
algo.fit(trainset_full)
train_time = time.time() - start_time

print(f"\n✅ Modèle final entraîné en {train_time:.1f}s")
print(f"   • Utilisateurs: {trainset_full.n_users:,}")
print(f"   • Films: {trainset_full.n_items:,}")

In [ ]:
# # 8. ÉVALUATION SUR TEST

# %%
print("\n📈 ÉVALUATION SUR TEST")
print("="*60)

# Prédictions sur test
predictions = algo.test(testset)

rmse = accuracy.rmse(predictions, verbose=False)
mae = accuracy.mae(predictions, verbose=False)

print(f"\n📊 RÉSULTATS FINAUX:")
print(f"   • RMSE: {rmse:.4f}")
print(f"   • MAE: {mae:.4f}")

# Interprétation
print(f"\n🔍 INTERPRÉTATION:")
if rmse < 0.85:
    print("   🏆 EXCELLENT! Modèle très précis")
elif rmse < 0.9:
    print("   ✅ TRÈS BON! Modèle performant")
elif rmse < 1.0:
    print("   👍 BON! Modèle acceptable")
else:
    print("   ⚠️ À AMÉLIORER")

# Visualisation des erreurs
errors = [pred.est - pred.r_ui for pred in predictions if pred.r_ui is not None]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(errors, bins=50, edgecolor='black')
axes[0].set_title('Distribution des erreurs')
axes[0].set_xlabel('Erreur (prédit - réel)')
axes[0].set_ylabel('Fréquence')
axes[0].axvline(0, color='r', linestyle='--')

# Prédictions vs Réel
real = [pred.r_ui for pred in predictions if pred.r_ui is not None]
pred = [pred.est for pred in predictions if pred.r_ui is not None]

axes[1].scatter(real, pred, alpha=0.1, s=1)
axes[1].plot([0.5, 5.0], [0.5, 5.0], 'r--')
axes[1].set_xlabel('Note réelle')
axes[1].set_ylabel('Note prédite')
axes[1].set_title('Prédictions vs Réel')

plt.tight_layout()
plt.show()

In [ ]:
# # 9. SAUVEGARDE DU MODÈLE

# %%
print("\n💾 SAUVEGARDE DU MODÈLE")
print("="*60)

# Sauvegarde locale
Path("models").mkdir(exist_ok=True)

model_data = {
    'model': algo,
    'best_params': best_params,
    'rmse': rmse,
    'mae': mae,
    'n_users': trainset_full.n_users,
    'n_items': trainset_full.n_items,
    'training_date': time.strftime("%Y-%m-%d %H:%M:%S"),
    'grid_search_results': results_df.head(10).to_dict()
}

with open("models/svd_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print(f"✅ Modèle sauvegardé: models/svd_model.pkl")

# Optionnel: sauvegarder sur Google Drive
with open("models/svd_model.pkl", "rb") as f:
    subprocess.run(
        ["rclone", "rcat", "gdrive:movie_data_25m/models/svd_model.pkl"],
        input=f.read()
    )

print("✅ Backup sur Google Drive")



In [ ]:
# # 10. TEST DU MODÈLE

# %%
print("\n🔍 TEST DU MODÈLE")
print("="*60)

# Tester quelques prédictions
for user_id in [0, 100, 500]:
    for item_id in [0, 50, 100]:
        pred = algo.predict(user_id, item_id)
        print(f"   • User {user_id}, Item {item_id}: {pred.est:.2f} ★")

# Recommandations pour un utilisateur
def get_recommendations(user_id, n=10):
    all_items = range(1000)  # Limité pour l'exemple
    predictions = []
    
    for item_id in all_items:
        pred = algo.predict(user_id, item_id)
        predictions.append((item_id, pred.est))
    
    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n]

print(f"\n🎯 Top 10 recommandations pour User 0:")
recommendations = get_recommendations(0)
for i, (item_id, score) in enumerate(recommendations, 1):
    print(f"   {i}. Item {item_id} → {score:.2f} ★")

In [ ]:
# # 11. CONCLUSION

# %%
print("\n" + "="*60)
print("✅ PIPELINE COMPLET RÉUSSI!")
print("="*60)
print(f"""
📊 RÉSUMÉ DU PROJET:
   • Données: {len(df_ratings):,} ratings
   • Utilisateurs: {trainset_full.n_users:,}
   • Films: {trainset_full.n_items:,}
   • Meilleurs paramètres:
     - n_factors: {best_params['n_factors']}
     - n_epochs: {best_params['n_epochs']}
     - lr_all: {best_params['lr_all']}
     - reg_all: {best_params['reg_all']}
   • Performance:
     - RMSE: {rmse:.4f}
     - MAE: {mae:.4f}

🎯 PROCHAINES ÉTAPES:
   1. API FastAPI pour servir le modèle
   2. Dashboard Streamlit pour visualisation
   3. Déploiement sur le cloud
""")
print("="*60)